<a href="https://colab.research.google.com/github/ced-sys/AI-N-ML/blob/main/Old_Assyrian_Gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from pathlib import Path
from tqdm.auto import tqdm
from typing import List, Dict, Tuple
import warnings
import pickle
warnings.filterwarnings('ignore')

In [ ]:
!pip install evaluate

In [ ]:
from transformers import(
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import evaluate
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def load_data_from_drive(
    drive_zip_path:str="/content/drive/MyDrive/Data Folder/data.zip",
    extract_to: Path=Path('/content/data')
)-> Path:

  from google.colab import drive as colab_drive

  mount_point=Path("/content/drive")
  if not(mount_point/ "MyDrive").exists():
    print("\nMounting Google Drive...")
    colab_drive.mount(str(mount_point))
  else:
    print("\nGoogle Drive already mounted.")

  zip_path=mount_point/drive_zip_path

  if not zip_path.exists():
    raise FileNotFoundError(
        f"\nCould not find zip at: {zip_path}"
    )

  print(f"\nFound: {zip_path}")
  size_mb=zip_path.stat().st_size/ (1024* 1024)
  print(f"Size: {size_mb:.1f} MB")

  extract_to.mkdir(exist_ok=True, parents=True)
  print(f'Extracting to {extract_to}...')

  with zipfile.ZipFile(zip_path, "r") as zf:
    members=zf.namelist()
    print(f" {len(members)} files in archive")
    for member in tqdm(members, desc="Extracting"):
      zf.extract(member, extract_to)


  top_level=list(extract_to.iterdir())
  if len(top_level)==1 and top_level[0].is_dir():
    subdir=top_level[0]
    print(f"\nFlattening subdirectory: {subdir.name}/")
    for f in subdir.iterdir():
      f.rename(extract_to / f.name)
    subdir.rmdir()

  required=[
      "train.csv",
      "test.csv",
      "published_texts.csv",
      "publications.csv",
      "OA_Lexicon_eBL.csv",
      "Sentences_Oare_FirstWord_LinNum.csv",
  ]
  missing=[]
  for fname in required:
    fpath=extract_to/fname
    if fpath.exists():
      mb=fpath.stat().st_size/ (1024*1024)
      print(f" OK {fname:<40s} ({mb:>7.1f}) MB")
    else:
      missing.append(fname)
      print(f" -- {fname:<40s} (MISSING)")

  if missing:
    print(f"WARNING: {len(missing)} file(s) missing: {', '.join(missing)}")
  else:
    print(f"ALL {len(required)} required files found.")

  return extract_to

In [ ]:
DRIVE_ZIP_PATH='/content/drive/MyDrive/Data Folder/data.zip'

In [ ]:
DATA_DIR=Path('/content/data')
OUTPUT_DIR=Path('/content/drive/MyDrive/old_assyrian_models')
KAGGLE_EXPORT_DIR=Path('/content/drive/MyDrive/old_assyrian_kaggle')

MODEL_NAME="google/gemma-2-2b-it"
MAX_LENGTH=512

NUM_EPOCHS=3
BATCH_SIZE=4
GRADIENT_ACCUMULATION_STEPS=4
LEARNING_RATE=2e-4
LORA_RANK=16
LORA_ALPHA=32

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
KAGGLE_EXPORT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
class AkkadianNormalizer:

  def __init__(self):
    self.vowel_map={
        '\u00e1':'a2',
        '\u00e0':'a3',
        '\u00e2':'a2',
        '\u00e9':'e2',
        '\u00e8':'e3',
        '\u00ea':'e2',
        '\u00ed':'i2',
        '\u00ec':'i3',
        '\u00ee':'i2',
        '\u00fa':'u2',
        '\u00f9':'u3',
        '\u00fb':'u2',
    }

    self.consonant_map={
        '\u0161':'sz',
        '\u0160':'SZ',
        '\u1e63':'S',
        '\u1e62':'S',
        '\u1e6d':'t',
        '\u1e6c':'T',
        '\u1e2b':'h',
        '\u1e2a':'H',
    }

    self.bracket_patterns=[
        (r'\u02f9([^\u02fa]+)\u02fa', r'\1'),
        (r'<([^>]+)>', r'\1'),
        (r'<<[^>]*>>', ''),
        (r'\[([^\]]+)\]', r'\1')
    ]

    self.lacuna_patterns=[
        (r'\[\.\.\.\s*\.\.\.\]', '[...]'),
        (r'\[\s*x\s*\]', '[x]'),
        (r'\.\.\.', '[...]'),
        (r'\[([^\]]{2,})\]', '[...]'),
        (r'\[([a-z0-9])\]', '[x]')
    ]

  def normalize_diacritics(self, text: str)-> str:
    for old, new in {**self.vowel_map, **self.consonant_map}.items():
      text=text.replace(old, new)
    return text

  def clean_brackets(self, text: str)->str:
    for pattern, replacement in self.bracket_patterns:
      text=re.sub(pattern, replacement, text)
    return text

  def standardize_lacunae(self, text: str)->str:
    for pattern, replacement in self.lacuna_patterns:
      text=re.sub(pattern, replacement, text)
    return text

  def remove_line_numbers(self, text: str)->str:
    text=re.sub(r'^\d+\'*\s*', '', text, flags=re.MULTILINE)
    text=re.sub(r'\s+\d+\'*\s*', '', text)
    return text

  def clean_punctuation(self, text: str)-> str:
    text=re.sub(r'[!?/]', '', text)
    text=re.sub(r'(?<!\d):(?!\d)', '', text)
    return text

  def normalize(self, text: str)->str:
    if pd.isna(text):
      return ""

    text=str(text)
    text=self.normalize_diacritics(text)
    text=self.clean_brackets(text)
    text=self.standardize_lacunae(text)
    text=self.remove_line_numbers(text)
    text=self.clean_punctuation(text)

    text=re.sub(r'\s+', '', text)
    text=text.strip()

    return text

In [ ]:
normalizer=AkkadianNormalizer()

test_text="a-na {d}A\u0161\u0161ur \u00e1-bi-ia q\u00ed-b\u00ed-ma [... ...] \u0161u-ma"
print(f"Original: {test_text}")
print(f"Normalized: {normalizer.normalize(test_text)}")

In [ ]:
def load_and_preprocess_training_data(data_dir: Path)-> pd.DataFrame:
  train_df=pd.read_csv(data_dir / 'train.csv')
  print(f"Training samples: {len(train_df)}")

  train_df['transliteration_clean']=train_df['transliteration'].apply(normalizer.normalize)
  train_df['translation_clean']=train_df['translation'].fillna('')

  train_df=train_df[train_df['translation_clean'].str.len()>0]
  print(f"After cleaning: {len(train_df)} samples")

  sample_idx=0
  print("CLEANED SAMPLE")
  print(f"Old Assyrian: {train_df.iloc[sample_idx]['transliteration_clean'][:200]}...")
  print(f"English: {train_df.iloc[sample_idx]['translation_clean'][:200]}...")

  return train_df

In [ ]:
def extract_translation_pairs(page_text: str, max_pairs: int=10)-> List[Tuple[str, str]]:
  pairs=[]
  lines=page_text.split('\n')

  i=0
  while i<len(lines)-1 and len(pairs)<max_pairs:
    line=lines[i].strip()

    if ('-' in line and len(line.split())> 2 and re.search(r'[a-z]{2,}-[a-z]{2,}', line)):
      translation=""
      for j in range(i+1, min(i+4, len(lines))):
        next_line=lines[j].strip()

        if (next_line and next_line[0].isupper() and '-' not in next_line):
          translation=next_line
          break

      if translation:
        akkadian=normalizer.normalize(line)
        pairs.append((akkadian, translation))
        i=j+1
        continue

    i+=1

  return pairs


In [ ]:
def extract_synthetic_data(data_dir: Path, num_pages: int=100)-> List[Tuple[str, str]]:
  try:
    publications_df=pd.read_csv(data_dir/ 'publications.csv')
    print(f"Publications: {len(publications_df)} pages")
    print(f"Pages with Akkadian: {publications_df['has_akkadian'].sum()}")

    synthetic_pairs=[]

    akkadian_pages=publications_df[publications_df['has_akkadian']==True].sample(
        min(num_pages, publications_df['has_akkadian'].sum()),
        random_state=42
    )

    print(f"Extracting pairs from {len(akkadian_pages)} pages...")
    for _, row in tqdm(akkadian_pages.iterrows(), total=len(akkadian_pages), desc="Extracting Pairs"):
      pairs=extract_translation_pairs(row['page_text'])
      synthetic_pairs.extend(pairs)

    print(f"Extracted {len(synthetic_pairs)} synthetic pairs")

    print("\nSynthetic pair samples:")
    for i, (akk, eng) in enumerate(synthetic_pairs[:3], 1):
      print(f"\n{i}.")
      print(f"OA: {akk[:100]}...")
      print(f" EN: {eng[:100]}...")

    return synthetic_pairs

  except FileNotFoundError:
    print("Publication files not found - using only train.csv")
    return []

In [ ]:
def create_training_dataset(train_df: pd.DataFrame, synthetic_pairs: List[Tuple[str, str]]) -> Tuple[List[Dict], List[Dict]]:
  training_data=[]

  for _, row in train_df.iterrows():
    training_data.append({
        'akkadian':row['transliteration_clean'],
        'english':row['translation_clean'],
        'source':'train.csv'
    })

  for akk, eng in synthetic_pairs:
    if len(akk.split())> 2 and len(eng.split())>2:
      training_data.append({
          'akkadian':akk,
          'english':eng,
          'source':'synthetic'
        })

  print(f"\nTotal training samples: {len(training_data)}")
  print(f" - Original: {sum(1 for d in training_data if d['source']=='train.csv')}")
  print(f" -Synthetic: {sum(1 for d in training_data if d['source']=='synthetic')}")

  train_data, val_data=train_test_split(
      training_data,
      test_size=0.1,
      random_state=42
  )

  print(f"\nTraining: {len(train_data)}")
  print(f"Validation: {len(val_data)}")

  return train_data, val_data

In [ ]:
def setup_model_and_tokenizer(model_name: str):

  print(f"\nLoading {model_name}.. in 4-bit (NF4)...")

  bnb_config=BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.float16,
      bnb_4bit_use_double_quant=True,
  )

  tokenizer=AutoTokenizer.from_pretrained(model_name)
  tokenizer.pad_token=tokenizer.eos_token
  tokenizer.padding_side='right'

  model=AutoModelForCausalLM.from_pretrained(
      model_name,
      quantization_config=bnb_config,
      device_map="auto",
  )

  model=prepare_model_for_kbit_training(model)

  lora_config=LoraConfig(
      r=LORA_RANK,
      lora_alpha=LORA_ALPHA,
      target_modules=["q_proj", "k_proj", 'v_proj', 'o_proj'],
      lora_dropout=0.05,
      bias="none",
      task_type="CAUSAL_LM"
  )

  model=get_peft_model(model, lora_config)
  model.print_trainable_parameters()

  print("Model ready!\n")
  return model, tokenizer

In [ ]:
def format_prompt(akkadian:str, englis: str=None, tokenizer=None)-> str:
  messages=[
      {
          "role":"user",
          "content":(
              "Translate this Old Assyrian cuneiform transliteration to English:\n\n"
              f"{akkadian}"
          )
      }
  ]

  if tokenizer is not None:
    prompt=tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    if english is not None:
      prompt+=f"{english}<end_of_turn>"
  else:
    prompt=(
        "<start_of_turn>user\n"
        "Translate this Old Assyrian cuneiform transliteratio to English:\n\n"
        f"{akkadian}<end_of_turn>\n"
        "<start_of_turn>\n"
    )
    if english is not None:
      prompt+=f"{english}<end_of_turn>"

  return prompt

In [ ]:
def preprocess_function(examples, tokenizer):
  prompts=[format_prompt(akk, eng) for akk, eng in zip(examples['akkadian'], examples['english'])]

  model_inputs=tokenizer(
      prompts,
      max_length=MAX_LENGTH,
      truncation=True,
      padding='max_length',
      return_tensors=None
  )

  model_inputs["labels"]=model_inputs['input_ids'].copy()
  return model_inputs

In [ ]:
def prepare_datasets(train_data: List[Dict], val_data: List[Dict], tokenizer):
  train_dataset=Dataset.from_list(train_data)
  val_dataset=Dataset.from_list(val_data)

  train_dataset=train_dataset.map(
      lambda x: preprocess_function(x, tokenizer),
      batched=True,
      remove_columns=val_dataset.column_names,
      desc='Tokenizing validation data'
  )

  val_dataset=val_dataset.map(
      lambda x: preprocess_function(x, tokenizer),
      batched=True,
      remove_columns=val_dataset.column_names,
      desc="Tokenisin validation data"
  )

  print(f"Train dataset: {len(train_dataset)} examples")
  print(f"Val dataset: {len(val_dataset)} examples")

  return train_dataset, val_dataset

In [ ]:
def create_trainer(model, tokenizer, train_dataset, val_dataset, output_dir: Path):
  training_args=TrainingArguments(
      output_dir=str(output_dir),

      num_train_epochs=NUM_EPOCHS,
      per_device_train_batch_size=BATCH_SIZE,
      per_device_eval_batch_size=BATCH_SIZE,
      gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

      learning_rate=LEARNING_RATE,
      warmup_steps=100,
      weight_decay=0.01,

      logging_steps=50,
      eval_strategy="steps",
      eval_steps=200,
      save_steps=200,
      save_total_limit=3,

      fp16=True,
      optim="paged_adamw_8bit",

      load_best_model_at_end=True,
      metric_for_best_model="eval_loss",
      report_to="none",
      push_to_hub=False,
  )

  data_collator=DataCollatorForLanguageModeling(
      tokenizer=tokenizer,
      mlm=False
  )

  trainer=Trainer(
     model=model,
     args=training_args,
     train_dataset=train_dataset,
     data_collator=data_collator,
  )

  return trainer

In [ ]:
!pip install sacrebleu

In [ ]:
bleu_metric=evaluate.load("bleu")
chrf_metric=evaluate.load("chrf")

In [ ]:
def compute_metrics(predictions: List[str], references: List[str])-> Dict[str, float]:
  bleu_result=bleu_metric.compute(
      predictions=predictions,
      references=[[ref] for ref in references]
  )

  bleu_score=bleu_result['bleu']*100

  chrf_result=chrf_metric.compute(
      predictions=predictions,
      references=references,
      word_order=2
  )
  chrf_score=chrf_result['score']

  geom_mean=np.sqrt(bleu_score*chrf_score)

  return{
      'bleu':bleu_score,
      'chrf':chrf_score,
      'geometric_mean':geom_mean
  }

In [ ]:
def generate_translation(akkadian_text: str, model, tokenizer, max_new_tokens: int=150)-> str:
  messages=[
      {
          "role":"user",
          "content":(
              "Translate this Old Assyrian cuneiform transliteration to English:\n\n"
              f"{akkadian_text}"
          )
      }
  ]

  inputs=tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt"
  ).to(model.device)

In [ ]:
def evaluate_on_validation(model, tokenizer, val_data: List[Dict], num_samples: int=50):
  val_sample=val_data[:num_samples]

  predictions=[]
  references=[]

  print("Generating validation predictions...")
  for sample in tqdm(val_sample):
    pred=generate_translation(sample['akkadian'], model, tokenizer)
    predictions.append(pred)
    references.append(sample['english'])

  metrics=compute_metrics(predictions, references)

  print(f"BLEU: {metrics['bleu']:.2f}")
  print(f"chrF++:  {metrics['chrf']:.2f}")
  print(f"Geometric Mean: {metrics['geometric_mean']:.2f}")

  for i in range(min(3, len(predictions))):
    print(f"\n{i+1}.")
    print(f"Old Assyrian: {val_sample[i]['akkadian'][:100]}...")
    print(f"Reference: {val_sample[i]['english'][:100]}...")
    print(f"Prediction: {predictions[i][:100]}...")

  return metrics

In [ ]:
def create_submission(model, tokenier, data_dir: Path, output_path: str='submission.csv'):
  test_df=pd.read_csv(data_dir /'test.csv')
  print(f"\nTest samples: {len(test_df)}")

  test_df['transliteration_clean']=test_df['tranliteration'].apply(normalizer.normalize)

  print("Generating test predictions...")
  test_predictions=[]

  for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    pred=generate_translation(row['transliteration_clean'], model, tokenizer)
    test_predictions.append(pred)

    submission_df=pd.DataFrame({
        'id':tet_df['id'],
        'translation':test_predictions
    })

    submission_df.to_csv(output_path, index=False)
    print(f"\nSubmission saved to {output_path}")

    empty=submission_df[submission_df['translation'].str.strip()=='']
    print(f"Empty translations: {len(empty)}")

    lengths=submission_df['translation'].str.len()
    print(f"\nTranslation length stats:")
    print(f" Min: {lengths.min()}")
    print(f" Max: {lengths.max()}")
    print(f" Mean: {lengths.mean():.1f}")
    print(f" Median: {lengths.median():.1f}")

    print("\nSample translations:")
    for i in range(min(3, len(submission_df))):
      row=submission_df.iloc[i]
      print(f"\n{i+1}. ID: {row['id']}")
      print(f" Translation: {row['translation'][:150]}...")

    return submission_df

In [ ]:
def main():
  data_dir=load_data_from_drive(extract_to=DATA_DIR)

  train_df=load_and_preprocess_training_data(data_dir)

  synthetic_pairs=extract_synthetic_data(data_dir, num_pages=100)

  train_data,val_data=create_training_dataset(train_df, synthetic_pairs)

  model, tokenizer=setup_model_and_tokenizer(MODEL_NAME)

  train_dataset, val_dataset=prepare_datasets(train_data, val_data, tokenizer)

  trainer=create_trainer(model, tokenizer, train_dataset, val_dataset, OUTPUT_DIR)

  trainer.train()

  final_model_path=OUTPUT_DIR / "final_model"
  trainer.save_model(final_model_path)
  tokenizer.save_pretrained(final_model_path)
  print(f"\nModel saved to {final_model_path}")

  evaluate_on_validation(model, tokenizer, val_data, num_samples=50)

  create_submission(model, tokenizer, data_dir)

In [ ]:
if __name__=="__main__":
  main()

In [ ]:
"""
Old Assyrian Machine Translation with Gemma
============================================

This script implements a complete pipeline for translating Old Assyrian
cuneiform transliterations to English using Google's Gemma models.

Strategy:
- Morphology-aware preprocessing for 4,000-year-old Akkadian
- LoRA fine-tuning of Gemma 2B/9B for efficiency
- Synthetic data generation from scholarly publications
- Character-level optimization (chrF++) for morphologically complex language

Workflow:
1. Data preprocessing & normalization
2. Synthetic training data extraction
3. Model fine-tuning with LoRA
4. Evaluation (BLEU × chrF++ geometric mean)
5. Export for Kaggle submission
"""

# ============================================================================
# 1. IMPORTS AND SETUP
# ============================================================================

import pandas as pd
import numpy as np
import re
import torch
from pathlib import Path
from tqdm.auto import tqdm
from typing import List, Dict, Tuple
import warnings
import pickle
import zipfile
import shutil
warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import evaluate
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ============================================================================
# 1.5 FILE UPLOAD AND UNZIP UTILITY
# ============================================================================

def load_data_from_drive(
    drive_zip_path: str = "MyDrive/old_assyrian/data.zip",
    extract_to: Path = Path('/content/data')
) -> Path:
    """
    Mount Google Drive and load competition data from it.

    This is much faster than uploading directly from your PC —
    Drive is already in Google's infrastructure so the transfer
    is nearly instant.

    Args:
        drive_zip_path : Path inside your Drive to the zip file,
                         relative to /content/drive/
                         e.g. "MyDrive/old_assyrian/data.zip"
        extract_to     : Local directory to extract the files into.

    Steps to set up (one time only):
        1. Go to drive.google.com
        2. Upload your competition zip there
        3. Note the path (e.g. MyDrive/Competitions/old_assyrian_data.zip)
        4. Set that as drive_zip_path below
    """
    from google.colab import drive as colab_drive

    print("=" * 80)
    print("LOADING DATA FROM GOOGLE DRIVE")
    print("=" * 80)

    # --- 1. Mount Drive ---
    mount_point = Path("/content/drive")
    if not (mount_point / "MyDrive").exists():
        print("\nMounting Google Drive...")
        colab_drive.mount(str(mount_point))
    else:
        print("\nGoogle Drive already mounted.")

    zip_path = mount_point / drive_zip_path

    # Debug: show what we're looking for
    print(f"\nLooking for zip at: {zip_path}")
    print(f"  Mount point: {mount_point}")
    print(f"  Drive path: {drive_zip_path}")
    print(f"  Full path: {zip_path}")
    print(f"  Path exists: {zip_path.exists()}")

    if not zip_path.exists():
        # Show what files ARE in the path to help debug
        parent_dir = zip_path.parent
        print(f"\n❌ Could not find zip at: {zip_path}")
        print(f"\nContents of {parent_dir}:")
        if parent_dir.exists():
            for item in sorted(parent_dir.iterdir())[:20]:
                size = ""
                if item.is_file():
                    size_mb = item.stat().st_size / (1024 * 1024)
                    size = f"  ({size_mb:.1f} MB)"
                print(f"  {'📁' if item.is_dir() else '📄'} {item.name}{size}")
        else:
            print(f"  Parent directory does not exist!")
        print(f"\n💡 Update DRIVE_ZIP_PATH to match your actual file.")
        print(f"   Current: {drive_zip_path}")
        raise FileNotFoundError(f"Zip not found at: {zip_path}")

    print(f"\nFound: {zip_path}")
    size_mb = zip_path.stat().st_size / (1024 * 1024)
    print(f"Size : {size_mb:.1f} MB")

    # --- 2. Extract ---
    extract_to.mkdir(exist_ok=True, parents=True)
    print(f"\nExtracting to {extract_to} ...")

    with zipfile.ZipFile(zip_path, "r") as zf:
        members = zf.namelist()
        print(f"  {len(members)} files in archive")
        for member in tqdm(members, desc="Extracting"):
            zf.extract(member, extract_to)

    # If everything landed inside a single subdirectory, flatten it
    top_level = list(extract_to.iterdir())
    if len(top_level) == 1 and top_level[0].is_dir():
        subdir = top_level[0]
        print(f"\nFlattening subdirectory: {subdir.name}/")
        for f in subdir.iterdir():
            f.rename(extract_to / f.name)
        subdir.rmdir()

    # --- 3. Verify ---
    print("\n" + "=" * 80)
    print("VERIFYING DATA FILES")
    print("=" * 80)

    required = [
        "train.csv",
        "test.csv",
        "published_texts.csv",
        "publications.csv",
        "OA_Lexicon_eBL.csv",
        "Sentences_Oare_FirstWord_LinNum.csv",
    ]
    missing = []
    for fname in required:
        fpath = extract_to / fname
        if fpath.exists():
            mb = fpath.stat().st_size / (1024 * 1024)
            print(f"  OK  {fname:<40s} ({mb:>7.1f} MB)")
        else:
            missing.append(fname)
            print(f"  --  {fname:<40s} (MISSING)")

    print("=" * 80)
    if missing:
        print(f"WARNING: {len(missing)} file(s) missing: {', '.join(missing)}")
    else:
        print(f"All {len(required)} required files found.")
    print("=" * 80 + "\n")

    return extract_to


# ============================================================================
# 2. CONFIGURATION
# ============================================================================

# Paths
DATA_DIR          = Path('/content/data')
OUTPUT_DIR        = Path('/content/drive/MyDrive/old_assyrian_models')
KAGGLE_EXPORT_DIR = Path('/content/drive/MyDrive/old_assyrian_kaggle')

# Google Drive path to your competition zip file (relative to /content/drive/)
# Example: "MyDrive/Competitions/old_assyrian_data.zip"
# Upload the zip to Google Drive once, then set this path and never upload again.
DRIVE_ZIP_PATH = "MyDrive/Data Folder/data.zip"  # <-- Update if your zip is in a different location

# Model settings
MODEL_NAME = "google/gemma-2-2b-it"  # or "google/gemma-2-9b-it"
MAX_LENGTH = 512

# Training settings
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
LORA_RANK = 16
LORA_ALPHA = 32

# Create directories
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
KAGGLE_EXPORT_DIR.mkdir(exist_ok=True, parents=True)

# ============================================================================
# 3. DATA PREPROCESSING: AKKADIAN NORMALIZER
# ============================================================================

class AkkadianNormalizer:
    """
    Comprehensive normalization for Old Assyrian transliterations.
    Converts scholarly notation to machine-readable ASCII format.

    Following the strategy document:
    - Diacritics → ASCII numeric notation (CDLI/ORACC convention)
    - Standardize lacunae markers ([x] and [...])
    - Clean scholarly annotations
    - Preserve determinatives and logograms

    Note: All special characters are represented as Unicode escapes (e.g., \u00e1)
    so the code can be typed on any standard keyboard.
    """

    def __init__(self):
        # Diacritic mapping (using Unicode escapes for special characters)
        # á = \u00e1, à = \u00e0, â = \u00e2, etc.
        self.vowel_map = {
            '\u00e1': 'a2',  # á - a with acute
            '\u00e0': 'a3',  # à - a with grave
            '\u00e2': 'a2',  # â - a with circumflex
            '\u00e9': 'e2',  # é - e with acute
            '\u00e8': 'e3',  # è - e with grave
            '\u00ea': 'e2',  # ê - e with circumflex
            '\u00ed': 'i2',  # í - i with acute
            '\u00ec': 'i3',  # ì - i with grave
            '\u00ee': 'i2',  # î - i with circumflex
            '\u00fa': 'u2',  # ú - u with acute
            '\u00f9': 'u3',  # ù - u with grave
            '\u00fb': 'u2',  # û - u with circumflex
        }

        self.consonant_map = {
            '\u0161': 'sz',  # š - s with caron (lowercase)
            '\u0160': 'SZ',  # Š - S with caron (uppercase)
            '\u1e63': 'S',   # ṣ - s with dot below (lowercase)
            '\u1e62': 'S',   # Ṣ - S with dot below (uppercase)
            '\u1e6d': 't',   # ṭ - t with dot below (lowercase)
            '\u1e6c': 'T',   # Ṭ - T with dot below (uppercase)
            '\u1e2b': 'h',   # ḫ - h with breve below (lowercase)
            '\u1e2a': 'H',   # Ḫ - H with breve below (uppercase)
        }

        # Bracket patterns (remove but preserve content)
        # Using Unicode escapes for special bracket characters
        self.bracket_patterns = [
            (r'\u02f9([^\u02fa]+)\u02fa', r'\1'),  # Half brackets ˹ ˺
            (r'<([^>]+)>', r'\1'),   # Pointy brackets
            (r'<<[^>]*>>', ''),      # Double pointy - remove entirely
            (r'\[([^\]]+)\]', r'\1') # Square brackets
        ]

        # Lacunae standardization
        self.lacuna_patterns = [
            (r'\[\.\.\.\s*\.\.\.\]', '[...]'),
            (r'\[\s*x\s*\]', '[x]'),
            (r'\.\.\.', '[...]'),
            (r'\[([^\]]{2,})\]', '[...]'),  # Multi-char gaps
            (r'\[([a-z0-9])\]', '[x]')      # Single char gaps
        ]

    def normalize_diacritics(self, text: str) -> str:
        """Convert diacritics to numeric notation."""
        for old, new in {**self.vowel_map, **self.consonant_map}.items():
            text = text.replace(old, new)
        return text

    def clean_brackets(self, text: str) -> str:
        """Remove or standardize bracket notations."""
        for pattern, replacement in self.bracket_patterns:
            text = re.sub(pattern, replacement, text)
        return text

    def standardize_lacunae(self, text: str) -> str:
        """Standardize all gap indicators to [x] or [...]."""
        for pattern, replacement in self.lacuna_patterns:
            text = re.sub(pattern, replacement, text)
        return text

    def remove_line_numbers(self, text: str) -> str:
        """Remove line numbers like 1, 1', 1'', etc."""
        # Remove line numbers at start of lines
        text = re.sub(r'^\d+\'*\s*', '', text, flags=re.MULTILINE)
        text = re.sub(r'\s+\d+\'*\s*', ' ', text)
        return text

    def clean_punctuation(self, text: str) -> str:
        """Remove modern punctuation except determinatives."""
        # Preserve {determinatives} but remove other punct
        text = re.sub(r'[!?/]', '', text)
        text = re.sub(r'(?<!\d):(?!\d)', '', text)  # Remove : but not in numbers
        return text

    def normalize(self, text: str) -> str:
        """Full normalization pipeline."""
        if pd.isna(text):
            return ""

        text = str(text)
        text = self.normalize_diacritics(text)
        text = self.clean_brackets(text)
        text = self.standardize_lacunae(text)
        text = self.remove_line_numbers(text)
        text = self.clean_punctuation(text)

        # Clean up whitespace
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()

        return text

# Initialize normalizer
normalizer = AkkadianNormalizer()

# Test the normalizer
# Using Unicode escapes: \u0161 = š, \u00e1 = á, etc.
test_text = "a-na {d}A\u0161\u0161ur \u00e1-bi-ia q\u00ed-b\u00ed-ma [... ...] \u0161u-ma"
print("\n" + "="*80)
print("NORMALIZER TEST:")
print(f"Original:   {test_text}")
print(f"Normalized: {normalizer.normalize(test_text)}")
print("="*80 + "\n")

# ============================================================================
# 4. DATA LOADING AND PREPROCESSING
# ============================================================================

def load_and_preprocess_training_data(data_dir: Path) -> pd.DataFrame:
    """Load and preprocess the training data."""
    print("Loading training data...")
    train_df = pd.read_csv(data_dir / 'train.csv')
    print(f"Training samples: {len(train_df)}")

    # Normalize transliterations
    print("Normalizing transliterations...")
    train_df['transliteration_clean'] = train_df['transliteration'].apply(normalizer.normalize)
    train_df['translation_clean'] = train_df['translation'].fillna('')

    # Remove empty translations
    train_df = train_df[train_df['translation_clean'].str.len() > 0]
    print(f"After cleaning: {len(train_df)} samples")

    # Display sample
    sample_idx = 0
    print("\n" + "="*80)
    print("CLEANED SAMPLE:")
    print(f"Old Assyrian: {train_df.iloc[sample_idx]['transliteration_clean'][:200]}...")
    print(f"English: {train_df.iloc[sample_idx]['translation_clean'][:200]}...")
    print("="*80 + "\n")

    return train_df

# ============================================================================
# 5. SYNTHETIC DATA EXTRACTION
# ============================================================================

def extract_translation_pairs(page_text: str, max_pairs: int = 10) -> List[Tuple[str, str]]:
    """
    Extract Old Assyrian -> English translation pairs from OCR text.

    Looks for patterns like:
    - Transliteration lines (contain hyphens, lowercase akkadian)
    - Followed by translation lines (English text)
    """
    pairs = []
    lines = page_text.split('\n')

    i = 0
    while i < len(lines) - 1 and len(pairs) < max_pairs:
        line = lines[i].strip()

        # Check if this looks like Akkadian transliteration
        if ('-' in line and
            len(line.split()) > 2 and
            re.search(r'[a-z]{2,}-[a-z]{2,}', line)):

            # Look for translation in next few lines
            translation = ""
            for j in range(i+1, min(i+4, len(lines))):
                next_line = lines[j].strip()
                # Translation likely starts with capital, no hyphens
                if (next_line and
                    next_line[0].isupper() and
                    '-' not in next_line):
                    translation = next_line
                    break

            if translation:
                akkadian = normalizer.normalize(line)
                pairs.append((akkadian, translation))
                i = j + 1
                continue

        i += 1

    return pairs

def extract_synthetic_data(data_dir: Path, num_pages: int = 100) -> List[Tuple[str, str]]:
    """Extract synthetic training pairs from scholarly publications."""
    try:
        publications_df = pd.read_csv(data_dir / 'publications.csv')
        print(f"Publications: {len(publications_df)} pages")
        print(f"Pages with Akkadian: {publications_df['has_akkadian'].sum()}")

        synthetic_pairs = []

        # Sample pages with Akkadian
        akkadian_pages = publications_df[publications_df['has_akkadian'] == True].sample(
            min(num_pages, publications_df['has_akkadian'].sum()),
            random_state=42
        )

        print(f"Extracting pairs from {len(akkadian_pages)} pages...")
        for _, row in tqdm(akkadian_pages.iterrows(), total=len(akkadian_pages), desc="Extracting pairs"):
            pairs = extract_translation_pairs(row['page_text'])
            synthetic_pairs.extend(pairs)

        print(f"Extracted {len(synthetic_pairs)} synthetic pairs")

        # Show samples
        print("\nSynthetic pair samples:")
        for i, (akk, eng) in enumerate(synthetic_pairs[:3], 1):
            print(f"\n{i}.")
            print(f"  OA: {akk[:100]}...")
            print(f"  EN: {eng[:100]}...")

        return synthetic_pairs

    except FileNotFoundError:
        print("Publication files not found - using only train.csv")
        return []

# ============================================================================
# 6. CREATE FINAL TRAINING DATASET
# ============================================================================

def create_training_dataset(train_df: pd.DataFrame, synthetic_pairs: List[Tuple[str, str]]) -> Tuple[List[Dict], List[Dict]]:
    """Combine original and synthetic data, then split train/val."""
    training_data = []

    # Add original training data
    for _, row in train_df.iterrows():
        training_data.append({
            'akkadian': row['transliteration_clean'],
            'english': row['translation_clean'],
            'source': 'train_csv'
        })

    # Add synthetic data (with quality filter)
    for akk, eng in synthetic_pairs:
        if len(akk.split()) > 2 and len(eng.split()) > 2:
            training_data.append({
                'akkadian': akk,
                'english': eng,
                'source': 'synthetic'
            })

    print(f"\nTotal training samples: {len(training_data)}")
    print(f"  - Original: {sum(1 for d in training_data if d['source'] == 'train_csv')}")
    print(f"  - Synthetic: {sum(1 for d in training_data if d['source'] == 'synthetic')}")

    # Split train/validation
    train_data, val_data = train_test_split(
        training_data,
        test_size=0.1,
        random_state=42
    )

    print(f"\nTraining: {len(train_data)}")
    print(f"Validation: {len(val_data)}")

    return train_data, val_data

# ============================================================================
# 7. MODEL SETUP: GEMMA + LORA
# ============================================================================

def setup_model_and_tokenizer(model_name: str):
    """
    Load Gemma model with 4-bit quantization + LoRA.

    RAM comparison for gemma-2-2b:
      float32 (no quant) : ~16 GB  -- kills Colab instantly
      float16            : ~8  GB  -- borderline on free tier
      4-bit NF4          : ~2  GB  -- what we use, safe on free Colab
    """
    from transformers import BitsAndBytesConfig

    print(f"\nLoading {model_name} in 4-bit (NF4)...")

    # 4-bit quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",           # NormalFloat4 - best quality at 4-bit
        bnb_4bit_compute_dtype=torch.float16, # Run compute in fp16 for speed
        bnb_4bit_use_double_quant=True,       # Nested quant saves ~0.4 GB extra
    )

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Load model in 4-bit
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )

    # Prepare for LoRA training
    model = prepare_model_for_kbit_training(model)

    # LoRA configuration
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    # Apply LoRA
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    print("Model ready!\n")
    return model, tokenizer

# ============================================================================
# 8. DATA FORMATTING AND TOKENIZATION
# ============================================================================

def format_prompt(akkadian: str, english: str = None, tokenizer=None) -> str:
    """
    Format a training example using apply_chat_template (Gemma's official method).

    apply_chat_template handles the <start_of_turn> / <end_of_turn> tokens
    automatically and correctly, which is cleaner and safer than building
    the string manually.

    Falls back to a plain string if tokenizer is not passed.
    """
    messages = [
        {
            "role": "user",
            "content": (
                "Translate this Old Assyrian cuneiform transliteration to English:\n\n"
                f"{akkadian}"
            )
        }
    ]

    if tokenizer is not None:
        # Use Gemma's official chat template
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,  # Appends <start_of_turn>model\n
            tokenize=False,              # Return string, not token IDs
        )
        if english is not None:
            prompt += f"{english}<end_of_turn>"
    else:
        # Plain fallback (no tokenizer available)
        prompt = (
            "<start_of_turn>user\n"
            "Translate this Old Assyrian cuneiform transliteration to English:\n\n"
            f"{akkadian}<end_of_turn>\n"
            "<start_of_turn>model\n"
        )
        if english is not None:
            prompt += f"{english}<end_of_turn>"

    return prompt

def preprocess_function(examples, tokenizer):
    """Tokenize examples for training."""
    prompts = [format_prompt(akk, eng, tokenizer=tokenizer) for akk, eng in zip(examples['akkadian'], examples['english'])]

    model_inputs = tokenizer(
        prompts,
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length',
        return_tensors=None
    )

    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

def prepare_datasets(train_data: List[Dict], val_data: List[Dict], tokenizer):
    """Create and tokenize HuggingFace datasets."""
    # Create datasets
    train_dataset = Dataset.from_list(train_data)
    val_dataset = Dataset.from_list(val_data)

    # Tokenize
    train_dataset = train_dataset.map(
        lambda x: preprocess_function(x, tokenizer),
        batched=True,
        remove_columns=train_dataset.column_names,
        desc="Tokenizing training data"
    )

    val_dataset = val_dataset.map(
        lambda x: preprocess_function(x, tokenizer),
        batched=True,
        remove_columns=val_dataset.column_names,
        desc="Tokenizing validation data"
    )

    print(f"Train dataset: {len(train_dataset)} examples")
    print(f"Val dataset: {len(val_dataset)} examples")

    return train_dataset, val_dataset

# ============================================================================
# 9. TRAINING CONFIGURATION
# ============================================================================

def create_trainer(model, tokenizer, train_dataset, val_dataset, output_dir: Path):
    """Create HuggingFace Trainer."""
    training_args = TrainingArguments(
        output_dir=str(output_dir),

        # Training params
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

        # Optimization
        learning_rate=LEARNING_RATE,
        warmup_steps=100,
        weight_decay=0.01,

        # Logging & evaluation
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=200,
        save_steps=200,
        save_total_limit=3,

        # Performance
        fp16=True,
        optim="paged_adamw_8bit",

        # Other
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
        push_to_hub=False,
    )

    # Data collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )

    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
    )

    return trainer

# ============================================================================
# 10. EVALUATION METRICS
# ============================================================================

# Load metrics
bleu_metric = evaluate.load("bleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(predictions: List[str], references: List[str]) -> Dict[str, float]:
    """
    Compute BLEU, chrF++, and their geometric mean.

    Evaluation metric: sqrt(BLEU × chrF++)
    """
    # BLEU
    bleu_result = bleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references]
    )
    bleu_score = bleu_result['bleu'] * 100

    # chrF++
    chrf_result = chrf_metric.compute(
        predictions=predictions,
        references=references,
        word_order=2  # chrF++ uses word order
    )
    chrf_score = chrf_result['score']

    # Geometric mean
    geom_mean = np.sqrt(bleu_score * chrf_score)

    return {
        'bleu': bleu_score,
        'chrf': chrf_score,
        'geometric_mean': geom_mean
    }

# ============================================================================
# 11. INFERENCE
# ============================================================================

def generate_translation(akkadian_text: str, model, tokenizer, max_new_tokens: int = 150) -> str:
    """
    Generate English translation from Old Assyrian text.

    Uses apply_chat_template + tokenize=True + return_dict=True (the same
    pattern as the HuggingFace Gemma quickstart) but with 4-bit model so
    RAM stays low.  We slice off the prompt tokens from the output so only
    the newly generated translation is decoded.
    """
    messages = [
        {
            "role": "user",
            "content": (
                "Translate this Old Assyrian cuneiform transliteration to English:\n\n"
                f"{akkadian_text}"
            )
        }
    ]

    # Tokenize with chat template - returns input_ids ready for generate()
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Slice off prompt tokens so we only decode the new generation
    prompt_length = inputs["input_ids"].shape[-1]
    translation = tokenizer.decode(
        outputs[0][prompt_length:],
        skip_special_tokens=True
    ).strip()

    return translation

def evaluate_on_validation(model, tokenizer, val_data: List[Dict], num_samples: int = 50):
    """Generate predictions and compute metrics on validation set."""
    val_sample = val_data[:num_samples]

    predictions = []
    references = []

    print("Generating validation predictions...")
    for sample in tqdm(val_sample):
        pred = generate_translation(sample['akkadian'], model, tokenizer)
        predictions.append(pred)
        references.append(sample['english'])

    # Compute metrics
    metrics = compute_metrics(predictions, references)

    print("\n" + "="*50)
    print("VALIDATION METRICS")
    print("="*50)
    print(f"BLEU:           {metrics['bleu']:.2f}")
    print(f"chrF++:         {metrics['chrf']:.2f}")
    print(f"Geometric Mean: {metrics['geometric_mean']:.2f}")
    print("="*50 + "\n")

    # Show some examples
    print("Sample predictions:")
    for i in range(min(3, len(predictions))):
        print(f"\n{i+1}.")
        print(f"Old Assyrian: {val_sample[i]['akkadian'][:100]}...")
        print(f"Reference:    {val_sample[i]['english'][:100]}...")
        print(f"Prediction:   {predictions[i][:100]}...")

    return metrics

# ============================================================================
# 12. TEST PREDICTION AND SUBMISSION
# ============================================================================

def create_submission(model, tokenizer, data_dir: Path, output_path: str = 'submission.csv'):
    """Generate predictions for test set and create submission file."""
    # Load test data
    test_df = pd.read_csv(data_dir / 'test.csv')
    print(f"\nTest samples: {len(test_df)}")

    # Normalize test transliterations
    test_df['transliteration_clean'] = test_df['transliteration'].apply(normalizer.normalize)

    # Generate predictions
    print("Generating test predictions...")
    test_predictions = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        pred = generate_translation(row['transliteration_clean'], model, tokenizer)
        test_predictions.append(pred)

    # Create submission
    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'translation': test_predictions
    })

    # Save
    submission_df.to_csv(output_path, index=False)
    print(f"\nSubmission saved to {output_path}")

    # Quality checks
    print("\n" + "="*50)
    print("SUBMISSION QUALITY CHECKS")
    print("="*50)

    # Check for empty translations
    empty = submission_df[submission_df['translation'].str.strip() == '']
    print(f"Empty translations: {len(empty)}")

    # Check translation lengths
    lengths = submission_df['translation'].str.len()
    print(f"\nTranslation length stats:")
    print(f"  Min: {lengths.min()}")
    print(f"  Max: {lengths.max()}")
    print(f"  Mean: {lengths.mean():.1f}")
    print(f"  Median: {lengths.median():.1f}")

    # Sample translations
    print(f"\nSample translations:")
    for i in range(min(3, len(submission_df))):
        row = submission_df.iloc[i]
        print(f"\n{i+1}. ID: {row['id']}")
        print(f"   Translation: {row['translation'][:150]}...")

    print("="*50 + "\n")

    return submission_df

# ============================================================================
# 13. EXPORT FOR KAGGLE
# ============================================================================

def export_for_kaggle(normalizer, output_dir: Path, export_dir: Path):
    """Package everything needed for Kaggle submission."""

    # 1. Save normalizer
    with open(export_dir / 'normalizer.pkl', 'wb') as f:
        pickle.dump(normalizer, f)
    print(f"Normalizer saved to {export_dir / 'normalizer.pkl'}")

    # 2. Model already saved in output_dir / "final_model"
    print(f"Model saved in: {output_dir / 'final_model'}")

    # 3. Create inference script
    inference_script = '''"""
Kaggle Inference Script for Old Assyrian Translation
"""

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import pickle
from tqdm import tqdm

# ============================================================================
# LOAD NORMALIZER
# ============================================================================

with open('/kaggle/input/your-dataset/normalizer.pkl', 'rb') as f:
    normalizer = pickle.load(f)

# ============================================================================
# LOAD MODEL
# ============================================================================

print("Loading model...")
base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b-it",
    load_in_4bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(base_model, "/kaggle/input/your-model/final_model")
tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/your-model/final_model")
model.eval()

# ============================================================================
# INFERENCE FUNCTION
# ============================================================================

def format_prompt(akkadian: str) -> str:
    prompt = f"""<start_of_turn>user
Translate this Old Assyrian cuneiform transliteration to English:

{akkadian}<end_of_turn>
<start_of_turn>model
"""
    return prompt

def generate_translation(akkadian_text: str, max_new_tokens: int = 150) -> str:
    prompt = format_prompt(akkadian_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)

    if "<start_of_turn>model" in full_output:
        translation = full_output.split("<start_of_turn>model")[-1]
        translation = translation.split("<end_of_turn>")[0].strip()
    else:
        translation = full_output

    return translation

# ============================================================================
# LOAD TEST DATA AND GENERATE PREDICTIONS
# ============================================================================

test_df = pd.read_csv('/kaggle/input/competition-data/test.csv')
print(f"Test samples: {len(test_df)}")

# Normalize
test_df['transliteration_clean'] = test_df['transliteration'].apply(normalizer.normalize)

# Generate predictions
predictions = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Translating"):
    pred = generate_translation(row['transliteration_clean'])
    predictions.append(pred)

# Create submission
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'translation': predictions
})

submission_df.to_csv('submission.csv', index=False)
print(f"\\nSubmission saved! Total predictions: {len(submission_df)}")
'''

    with open(export_dir / 'kaggle_inference.py', 'w') as f:
        f.write(inference_script)
    print(f"Inference script saved to {export_dir / 'kaggle_inference.py'}")

    print(f"\n{'='*50}")
    print("EXPORT COMPLETE!")
    print(f"{'='*50}")
    print(f"Files saved to: {export_dir}")
    print("\nNext steps:")
    print("1. Upload trained model to Kaggle as a dataset")
    print("2. Upload normalizer.pkl to Kaggle")
    print("3. Use kaggle_inference.py in your submission notebook")
    print(f"{'='*50}\n")

# ============================================================================
# 14. MAIN EXECUTION
# ============================================================================

def main():
    """Main execution pipeline."""

    print("="*80)
    print("OLD ASSYRIAN MACHINE TRANSLATION WITH GEMMA")
    print("="*80 + "\n")

    # 0. Load data from Google Drive (much faster than uploading from PC)
    data_dir = load_data_from_drive(drive_zip_path=DRIVE_ZIP_PATH, extract_to=DATA_DIR)

    # 1. Load and preprocess training data
    train_df = load_and_preprocess_training_data(data_dir)

    # 2. Extract synthetic data
    synthetic_pairs = extract_synthetic_data(data_dir, num_pages=100)

    # 3. Create final training dataset
    train_data, val_data = create_training_dataset(train_df, synthetic_pairs)

    # 4. Setup model and tokenizer
    model, tokenizer = setup_model_and_tokenizer(MODEL_NAME)

    # 5. Prepare datasets
    train_dataset, val_dataset = prepare_datasets(train_data, val_data, tokenizer)

    # 6. Create trainer
    trainer = create_trainer(model, tokenizer, train_dataset, val_dataset, OUTPUT_DIR)

    # 7. Train
    print("\n" + "="*80)
    print("STARTING TRAINING")
    print("="*80 + "\n")
    trainer.train()

    # 8. Save final model
    final_model_path = OUTPUT_DIR / "final_model"
    trainer.save_model(final_model_path)
    tokenizer.save_pretrained(final_model_path)
    print(f"\nModel saved to {final_model_path}")

    # 9. Evaluate on validation set
    print("\n" + "="*80)
    print("VALIDATION EVALUATION")
    print("="*80 + "\n")
    evaluate_on_validation(model, tokenizer, val_data, num_samples=50)

    # 10. Generate test predictions
    print("\n" + "="*80)
    print("GENERATING TEST PREDICTIONS")
    print("="*80 + "\n")
    create_submission(model, tokenizer, data_dir)

    # 11. Export for Kaggle
    print("\n" + "="*80)
    print("EXPORTING FOR KAGGLE")
    print("="*80 + "\n")
    export_for_kaggle(normalizer, OUTPUT_DIR, KAGGLE_EXPORT_DIR)

    print("\n" + "="*80)
    print("PIPELINE COMPLETE!")
    print("="*80 + "\n")

if __name__ == "__main__":
    main()